# Model 4: Disaster Type Classification

**Goal:** Identify the inherent disaster type (flood, volcano, fire, wind) directly from visual cues. Fast classification using MobileNetV2.

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

## 1. Network Architecture
MobileNetV2 provides excellent performance given strict speed requirements, which is great for Hackathon inference.

In [12]:
# Assuming 4 main disaster types based on demo folders
disaster_types = ['volcano', 'flooding', 'earthquake', 'fire', 'wind', 'tsunami']
num_classes = len(disaster_types)

base_model = MobileNetV2(
    include_top=False, 
    weights='imagenet', 
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# Construct Classification Head
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,358 (8.93 MB)

 Trainable params: 82,374 (321.77 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 2. Fast Training Loop
We parse the metadata to determine the macro disaster type.

In [14]:
import os, glob, json
import numpy as np

TRAIN_IMG_DIR = "../train/images"
DISASTERS = {"volcano": 0, "flooding": 1, "earthquake": 2, "fire": 3, "wind": 4, "tsunami": 5}
IMG_SIZE = (224, 224)

def get_disaster_type(img_path):
    img_path = img_path.numpy().decode('utf-8')
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    
    label_path = img_path.replace("images", "labels").replace(".png", ".json")
    lbl = 0
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            data = json.load(f)
        dtype = data['metadata'].get('disaster_type', 'volcano')
        lbl = DISASTERS.get(dtype, 0)
        
    return img, np.int32(lbl)

def tf_parse_type(img_path):
    img, lbl = tf.py_function(get_disaster_type, [img_path], [tf.float32, tf.int32])
    img.set_shape([224, 224, 3])
    lbl.set_shape([])
    return img, lbl

train_paths = glob.glob(os.path.join(TRAIN_IMG_DIR, "*.png"))
train_ds = tf.data.Dataset.from_tensor_slices(train_paths[:500])
train_ds = train_ds.map(tf_parse_type, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

print("Training Disaster Type Classifier...")
history = model.fit(train_ds, epochs=10)
model.save("disaster_type_model.h5")
print("Saved Model")


Training Disaster Type Classifier...
Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 471ms/step - accuracy: 0.6320 - loss: 0.9659
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 499ms/step - accuracy: 0.6180 - loss: 0.9801
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 593ms/step - accuracy: 0.6320 - loss: 0.9150
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 13s 762ms/step - accuracy: 0.6600 - loss: 0.8529
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 13s 778ms/step - accuracy: 0.6720 - loss: 0.8247
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 11s 679ms/step - accuracy: 0.6820 - loss: 0.8097
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.7120 - loss: 0.7668
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.7080 - loss: 0.7775
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.7260 - loss: 0.7606
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.7380 - loss: 0.7102


Saved Model
